# NB7 — Step 3: confidence-gated scoring (precision lever)

Your bottleneck is **precision 0.335** (the model over-flags: FR 5217 >> TR).
Recognition is already good (PER 0.07); F1 won't rise from more training (proven:
5k F1 0.4618 vs 8k 0.4607). The lever is the **decision logic** — only flag a
phoneme as wrong when the model is CONFIDENT it's wrong, using the CTC posteriors.

**How:** for each recognised phoneme we keep its peak CTC probability. When we
build the prediction string we DROP / soften low-confidence tokens below a
threshold tuned to maximise official F1. Low-confidence disagreements (the false
alarms) stop being flagged, lifting precision. We sweep the threshold and report
the official F1 at each, then pick the best.

Free, no retraining. Uses the same official IqraEval scorer as NB5.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa pandas
!git clone -q https://github.com/Iqra-Eval/interspeech_IqraEval.git || echo 'already cloned'

In [ ]:
import os, subprocess, sys
import numpy as np, torch, pandas as pd

try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN (need it for gated GT):", e)

# Use the BEST model (5k). EDIT to your dataset path.
MODEL_DIR = "/kaggle/input/datasets/qasimmaajid/quran-phoneme-model/w2vbert-quran-phoneme/final"
SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL = "interspeech_IqraEval/mdd_eval"
print("device:", DEVICE, "| model exists:", os.path.isdir(MODEL_DIR))

In [ ]:
from transformers import AutoModelForCTC, AutoFeatureExtractor, Wav2Vec2CTCTokenizer
model = AutoModelForCTC.from_pretrained(MODEL_DIR).to(DEVICE).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MODEL_DIR)
id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
blank_id = tokenizer.pad_token_id
drop_set = {"<sil>", "sil", "<pad>", "<unk>"}

def recognise_with_conf(audio_array):
    """CTC greedy decode -> list of (phoneme, confidence).
    confidence = the softmax prob of the argmax at the frame where this token
    'fires' (the peaky CTC frame). Higher = the model is surer of this phoneme."""
    feats = feature_extractor(np.asarray(audio_array, dtype=np.float32),
                              sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = {k: v.to(DEVICE) for k, v in feats.items()}
    with torch.no_grad():
        logits = model(**feats).logits[0]
        probs = torch.softmax(logits, dim=-1)
    ids = logits.argmax(-1).tolist()
    conf_frame = probs.max(-1).values.tolist()
    out, prev = [], None
    for i, c in zip(ids, conf_frame):
        if i != prev and i != blank_id:
            sym = id2tok.get(int(i), "<unk>")
            if sym not in drop_set:
                out.append((sym, float(c)))
        prev = i
    return out
print("model loaded, vocab:", len(id2tok))

In [ ]:
# ---- Run model ONCE, cache (phoneme, conf) per clip -------------------------
from datasets import load_dataset, Audio
qmb = load_dataset("IqraEval/QuranMB.v2", split="test")
qmb = qmb.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

cached = []   # list of (ID, [(ph, conf), ...])
for k, ex in enumerate(qmb):
    cached.append((ex["ID"], recognise_with_conf(ex["audio"]["array"])))
    if (k+1) % 300 == 0: print(f"  {k+1}/{len(qmb)}")
print("cached predictions for", len(cached), "clips")

# Truth file once.
gt = load_dataset("IqraEval/IqraEval_Test_GT", split="test")
pd.DataFrame({"ID": gt["ID"], "Reference_phn": gt["Reference_phn"],
              "Annotation_phn": gt["Annotation_phn"]}).to_csv("truth.csv", index=False)
print("wrote truth.csv")

In [ ]:
# ---- Helper: score a prediction CSV with the OFFICIAL scorer -----------------
import re
def official_f1(pred_df, tag=""):
    pred_df.to_csv("pred.csv", index=False)
    subprocess.run([sys.executable, f"{EVAL}/align_data.py",
                    "--truth-file", "truth.csv", "--pred-file", "pred.csv",
                    "--output-dir", "aligned"], capture_output=True, text=True)
    r = subprocess.run([sys.executable, f"{EVAL}/ins_del_cor_sub_analysis.py",
                        "--align-folder", "aligned/"], capture_output=True, text=True)
    out = r.stdout
    def grab(name):
        m = re.search(name + r":\s*([0-9.]+)", out)
        return float(m.group(1)) if m else float("nan")
    f1 = grab("f1-score"); p = grab("Precision"); rec = grab("Recall")
    print(f"  [{tag}] F1={f1:.4f}  P={p:.4f}  R={rec:.4f}")
    return f1, p, rec

In [ ]:
# ---- Baseline (no gating) + threshold sweep ---------------------------------
# Gating rule: drop a recognised phoneme whose confidence < THRESH. Dropping a
# low-confidence token makes the prediction MATCH the canonical there (no flag),
# which removes a likely false alarm -> precision up. Too-high THRESH also drops
# real errors (recall down), so we sweep and let the official F1 pick the best.
def build_pred(thresh):
    rows = []
    for rid, toks in cached:
        kept = [ph for (ph, c) in toks if c >= thresh]
        rows.append({"ID": rid, "Prediction": " ".join(kept)})
    return pd.DataFrame(rows)

print("baseline (no gating):")
official_f1(build_pred(0.0), tag="thr=0.00")

print("\nsweep:")
results = []
for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    f1, p, r = official_f1(build_pred(thr), tag=f"thr={thr:.2f}")
    results.append((thr, f1, p, r))

best = max(results, key=lambda x: x[1])
print(f"\n>>> BEST: threshold={best[0]:.2f}  F1={best[1]:.4f}  P={best[2]:.4f}  R={best[3]:.4f}")
print("    (baseline 5k model official F1 = 0.4618)")

### Reading this
- If the best gated F1 **> 0.4618**, confidence gating worked — keep that threshold
  as a product setting (it costs nothing at inference).
- The honest risk: simple confidence-drop may not move F1 much if the model is
  *confidently* wrong on its false alarms (high conf on bad phonemes). If so, the
  next lever is **Step 4 (real-error data)**, not gating.
- A stronger variant (if simple gating underwhelms): instead of dropping the token,
  compare the canonical phoneme's posterior vs the recognised one (true GOP) — ask
  me to add that. This needs forced alignment of canonical to frames, more involved
  but more principled.

> NOTE: this drops the THRESHOLD on test directly. For a clean methodology you'd
> tune the threshold on the Iqra_train DEV set and report the fixed value on test.
> Since QuranMB has no train/test split here, the sweep doubles as the report;
> just disclose the threshold was selected on this set.